# WordEmbedding 기반 텍스트 분석 연습
> [02_text_analysis.ipynb]의 develop

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_preprocessing.ipynb]
2. Word Embedding model을 이용하여 벡터화한다.
3. 입력된 문자열의 긍/부정을 판단한다. (유사도 활용)

In [6]:
# # 한국어 혐오 데이터셋
# from Korpora import Korpora
# Korpora.fetch("korean_hate_speech")

In [7]:
import pandas as pd
from pathlib import Path

base_dir = Path("C:/Users/Playdata/Korpora/korean_hate_speech")

all_comments = []

# labeled
for path in [
    base_dir / "labeled" / "train.tsv",
    base_dir / "labeled" / "dev.tsv"
]:
    if path.exists():
        df = pd.read_csv(path, sep="\t")
        if "comments" in df.columns:
            all_comments.extend(df["comments"].dropna().astype(str).tolist())

# unlabeled 폴더 전체 읽기
unlabeled_dir = base_dir / "unlabeled"

for file_path in unlabeled_dir.glob("*"):
    try:
        if file_path.suffix == ".tsv":
            df = pd.read_csv(file_path, sep="\t")
            if "comments" in df.columns:
                all_comments.extend(df["comments"].dropna().astype(str).tolist())
        elif file_path.suffix == ".csv":
            df = pd.read_csv(file_path)
            if "comments" in df.columns:
                all_comments.extend(df["comments"].dropna().astype(str).tolist())
        elif file_path.suffix == ".txt":
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_comments.append(line)
    except Exception as e:
        print(f"읽기 실패: {file_path} / {e}")

# DataFrame으로 정리
corpus_df = pd.DataFrame({"comments": all_comments})

corpus_df = corpus_df.dropna()
corpus_df["comments"] = corpus_df["comments"].astype(str).str.strip()
corpus_df = corpus_df[corpus_df["comments"] != ""]
corpus_df = corpus_df.drop_duplicates()

print(corpus_df.shape) # 1850053개로 너무 많음

# 2만개 랜덤 추출로 데이터 양 줄이기
corpus_df = corpus_df.sample(n=20000, random_state=42)
print(corpus_df.shape)
print(corpus_df.head())

(1850053, 1)
(20000, 1)
                                                  comments
190256   시청율은 몰라도 우선 1박2일의 존제 목적에 대해서는 가장 잘 보여주고 있다고 봅니...
562126                                         이가은 떨어지나보네욤
1080472                                 세기의결혼???아무거나 갔다붙이네
1814771                            연정훈 탈모인가보네 ㅜㅜ 은근 다 재밌네요
995798                 사이다는 없는데요....뭐 마지막회에 몰아서 다 하려나...답답


In [8]:
import re

def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)       # 공백 정리
    text = re.sub(r"[^\w\s가-힣ㄱ-ㅎㅏ-ㅣ]", " ", text)  # 특수문자 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

corpus_df["clean_comments"] = corpus_df["comments"].apply(clean_text)

# 너무 짧은 문장 제거
corpus_df = corpus_df[corpus_df["clean_comments"].str.len() >= 2]

print(corpus_df[["comments", "clean_comments"]].head())

                                                  comments  \
190256   시청율은 몰라도 우선 1박2일의 존제 목적에 대해서는 가장 잘 보여주고 있다고 봅니...   
562126                                         이가은 떨어지나보네욤   
1080472                                 세기의결혼???아무거나 갔다붙이네   
1814771                            연정훈 탈모인가보네 ㅜㅜ 은근 다 재밌네요   
995798                 사이다는 없는데요....뭐 마지막회에 몰아서 다 하려나...답답   

                                            clean_comments  
190256   시청율은 몰라도 우선 1박2일의 존제 목적에 대해서는 가장 잘 보여주고 있다고 봅니...  
562126                                         이가은 떨어지나보네욤  
1080472                                   세기의결혼 아무거나 갔다붙이네  
1814771                            연정훈 탈모인가보네 ㅜㅜ 은근 다 재밌네요  
995798                      사이다는 없는데요 뭐 마지막회에 몰아서 다 하려나 답답  


### 토크나이징 하기

In [9]:
from konlpy.tag import Okt

okt = Okt() # 형태소 분석기 객체 생성

# 한국어 불용어 파일
with open('./etc/ko_stopwords.txt', 'r', encoding='UTF-8') as f:
    ko_stopwords =  [line.strip() for line in f]

tokenized_corpus = []

for text in corpus_df["clean_comments"]:
    tokens = okt.morphs(text, stem=True)
    tokens = [t for t in tokens if len(t) > 1]  # 한 글자 제거는 선택
    tokens = [token for token in tokens if token not in ko_stopwords]
    if tokens:
        tokenized_corpus.append(tokens)

print(tokenized_corpus[:3])
print(len(tokenized_corpus))

[['시청율', '모르다', '2일', '존제', '목적', '대해', '서다', '가장', '자다', '보여주다', '보다', '이전', '까지만', '해도', '시청율', '높다', '자극', '거나', '반복', '내용', '안보', '최근', '방송', '되다', '2일', '자극', '이지', '않다', '정말', '좋다'], ['이가은', '떨어지다', '보다'], ['세기', '결혼', '아무', '거나', '가다', '붙이']]
19935


In [10]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1
)

In [11]:
model.wv.most_similar('행복',topn=100)

[('건강', 0.9881097674369812),
 ('홧팅', 0.9825215935707092),
 ('챙기다', 0.9808928966522217),
 ('가득하다', 0.9784798622131348),
 ('다행', 0.9778324961662292),
 ('알콩달콩', 0.9757128357887268),
 ('승승장구', 0.9741689562797546),
 ('고맙다', 0.9739537239074707),
 ('일만', 0.9738337993621826),
 ('기억', 0.9734621047973633),
 ('오래오래', 0.9733109474182129),
 ('축복', 0.9725978970527649),
 ('얼른', 0.9724101424217224),
 ('아내', 0.9713408946990967),
 ('다녀오다', 0.9710580110549927),
 ('감사하다', 0.9696362614631653),
 ('느껴지다', 0.9694716930389404),
 ('어머니', 0.967679500579834),
 ('마무리', 0.9662379622459412),
 ('화목', 0.9649574756622314),
 ('최선', 0.9645955562591553),
 ('언제나', 0.9640759229660034),
 ('이루다', 0.9632418155670166),
 ('기도', 0.9627745747566223),
 ('오래', 0.9626983404159546),
 ('꽃길', 0.9620429277420044),
 ('밝다', 0.9603123664855957),
 ('이에요', 0.9578939080238342),
 ('계시다', 0.9571341872215271),
 ('배려', 0.9557397961616516),
 ('아끼다', 0.9555132389068604),
 ('관리', 0.9549857974052429),
 ('천사', 0.9540314078330994),
 ('기쁘다', 0.9536890983

In [12]:
model.wv.most_similar('불행',topn=100)

[('고등학교', 0.996307373046875),
 ('여인', 0.9961825013160706),
 ('싱글맘', 0.9959801435470581),
 ('집안일', 0.9958221316337585),
 ('여유', 0.9957068562507629),
 ('성숙하다', 0.9955995082855225),
 ('버릇', 0.9955905079841614),
 ('국제', 0.9954832792282104),
 ('사회생활', 0.9954788088798523),
 ('파탄', 0.9954419136047363),
 ('이전', 0.9953660368919373),
 ('개도', 0.9953369498252869),
 ('만지다', 0.9952834844589233),
 ('경험', 0.9952335357666016),
 ('어려움', 0.9951879382133484),
 ('진리', 0.9951034784317017),
 ('바람나다', 0.9950598478317261),
 ('위자료', 0.9949998259544373),
 ('감당', 0.9949015378952026),
 ('농구', 0.994871973991394),
 ('당첨', 0.9948556423187256),
 ('루저', 0.9948393106460571),
 ('정석', 0.9948113560676575),
 ('과는', 0.9948102831840515),
 ('알려지다', 0.994773268699646),
 ('점도', 0.9947174191474915),
 ('면전', 0.9947008490562439),
 ('적절하다', 0.9946632385253906),
 ('모시다', 0.9946034550666809),
 ('17년', 0.9945906400680542),
 ('착실하다', 0.9945904016494751),
 ('짐작', 0.9945371150970459),
 ('가정사', 0.9945282936096191),
 ('데려오다', 0.994488656520